#  Bangalore Traffic Analysis
## Notebook 2 — Data Cleaning

This notebook cleans and validates all ingested data:
- Handling nulls and duplicates
- Standardizing timestamps
- Validating coordinates
- Flagging outliers


### Setup

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "bangalore_traffic", "user": "postgres", "password": "prabhupada",
}
DB_URL = (f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
          f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}")
engine = create_engine(DB_URL)

BANGALORE_BBOX = {"lat_min": 12.834, "lat_max": 13.141, "lon_min": 77.460, "lon_max": 77.748}

def load(table):
    return pd.read_sql(f"SELECT * FROM {table}", engine)

print(" Connected to database.")


 Connected to database.


###  1. Traffic Speeds — Clean


In [2]:
df = load("traffic_speeds")
print(f"Shape before: {df.shape}")
print(df.isnull().sum())


Shape before: (7250, 11)
id                       0
junction_label           0
latitude                 0
longitude                0
fetched_at               0
current_speed            0
free_flow_speed          0
current_travel_time      0
free_flow_travel_time    0
confidence               0
road_closure             0
dtype: int64


In [3]:
# Drop rows where core speed values are missing
df.dropna(subset=["current_speed", "free_flow_speed"], inplace=True)

# Remove negative or zero speeds (data error)
df = df[(df["current_speed"] > 0) & (df["free_flow_speed"] > 0)]

# Cap unrealistic speeds (>150 km/h in Bangalore = sensor error)
df = df[df["current_speed"] <= 150]

# Parse timestamp
df["fetched_at"] = pd.to_datetime(df["fetched_at"])
df["hour"] = df["fetched_at"].dt.hour
df["day_of_week"] = df["fetched_at"].dt.day_name()
df["is_weekend"] = df["fetched_at"].dt.weekday >= 5

# Congestion index: how much slower than free-flow (0 = free, 1 = standstill)
df["congestion_index"] = 1 - (df["current_speed"] / df["free_flow_speed"])
df["congestion_index"] = df["congestion_index"].clip(0, 1)

# Add congestion label
def label_congestion(ratio):
    if ratio >= 0.75: return "Standstill"
    elif ratio >= 0.50: return "Heavy"
    elif ratio >= 0.25: return "Moderate"
    else: return "Free Flow"

df["congestion_level"] = df["congestion_index"].apply(label_congestion)

df.to_sql("traffic_speeds_clean", engine, if_exists="replace", index=False)
print(f"Shape after:  {df.shape}")
print(" traffic_speeds_clean saved.")
df.head(3)


Shape after:  (7250, 16)
 traffic_speeds_clean saved.


,id,junction_label,latitude,longitude,fetched_at,current_speed,free_flow_speed,current_travel_time,free_flow_travel_time,confidence,road_closure,hour,day_of_week,is_weekend,congestion_index,congestion_level
0,7261,Silk Board,12.9716,77.5946,2026-05-09 17:08:33.916585,18.0,26.0,1929,1335,1.0,False,17,Saturday,True,0.307692,Moderate
1,7262,Marathahalli,12.9352,77.6245,2026-05-09 17:08:34.638185,30.0,30.0,684,684,1.0,False,17,Saturday,True,0.000000,Free Flow
2,7263,Whitefield,12.9698,77.7499,2026-05-09 17:08:35.391307,27.0,38.0,1435,1020,1.0,False,17,Saturday,True,0.289474,Moderate


###  2. Weather Data — Clean


In [4]:
df = load("weather_data")
print(f"Shape before: {df.shape}")
print(df.isnull().sum())


Shape before: (20640, 6)
temperature_c       0
precipitation_mm    0
windspeed_kmh       0
weathercode         0
date                0
hour                0
dtype: int64


In [5]:
# Fill small gaps with forward fill (hourly data)
df.sort_values(["date", "hour"], inplace=True)
df[["temperature_c", "precipitation_mm", "windspeed_kmh"]] = (
    df[["temperature_c", "precipitation_mm", "windspeed_kmh"]].ffill()
)

# Clamp negatives for rainfall
df["precipitation_mm"] = df["precipitation_mm"].clip(lower=0)

# Label monsoon months
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.month
df["is_monsoon"] = df["month"].isin([6, 7, 8, 9])

# Rainfall category
def rain_category(mm):
    if mm == 0:       return "none"
    elif mm < 2.5:    return "light"
    elif mm < 7.5:    return "moderate"
    elif mm < 15.0:   return "heavy"
    else:             return "very_heavy"

df["rain_category"] = df["precipitation_mm"].apply(rain_category)

# Season category
df["season"] = df["month"].apply(
    lambda m: "Monsoon" if m in [6,7,8,9] else ("Summer" if m in [3,4,5] else "Winter/Dry")
)

df.to_sql("weather_clean", engine, if_exists="replace", index=False)
print(f"Shape after:  {df.shape}")
print(" weather_clean saved.")
df.head(3)


Shape after:  (20640, 10)
 weather_clean saved.


,temperature_c,precipitation_mm,windspeed_kmh,weathercode,date,hour,month,is_monsoon,rain_category,season
0,17.6,0.0,14.2,0,2024-01-01,0,1,False,none,Winter/Dry
1,16.9,0.0,14.4,1,2024-01-01,1,1,False,none,Winter/Dry
2,16.6,0.0,14.4,3,2024-01-01,2,1,False,none,Winter/Dry


###  3. Accidents — Clean


In [6]:
df = load("accidents")
print(f"Shape before: {df.shape}")
print(df.isnull().sum())


Shape before: (10, 11)
date              0
time              0
location          0
latitude          0
longitude         0
severity          0
vehicles          0
casualties        0
violation_type    0
fatalities        0
injuries          0
dtype: int64


In [7]:
# Only write to accidents_clean if enrich_data.py hasn't already populated it
with engine.connect() as _c:
    _n = _c.execute(text("SELECT COUNT(*) FROM accidents_clean")).scalar()

if _n >= 100:
    print(f"accidents_clean already has {_n} enriched rows — skipping overwrite.")
    print("To reset, run: enrich_data.py")
else:
    # Drop rows with no location info at all
    df.dropna(subset=["location"], inplace=True)

    # Filter to valid Bangalore coordinates (where available)
    if "latitude" in df.columns and "longitude" in df.columns:
        bbox = BANGALORE_BBOX
        mask = (
            df["latitude"].between(bbox["lat_min"], bbox["lat_max"]) &
            df["longitude"].between(bbox["lon_min"], bbox["lon_max"])
        )
        invalid = (~mask & df["latitude"].notna()).sum()
        df = df[mask | df["latitude"].isna()]
        print(f"  Dropped {invalid} records outside Bangalore bbox.")

    # Standardize date
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df.dropna(subset=["date"], inplace=True)
        df["year"] = df["date"].dt.year
        df["month"] = df["date"].dt.month
        df["hour"] = pd.to_datetime(df["time"], format="%H:%M", errors="coerce").dt.hour

    # Numeric columns
    for col in ["vehicles", "casualties"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

    # Add a severity score for ranking
    if "severity" in df.columns:
        df["severity"] = df["severity"].fillna("Unknown").str.strip().str.title()
        severity_map = {"Fatal": 3, "Grievous": 2, "Minor": 1, "Unknown": 0}
        df["severity_score"] = df["severity"].map(severity_map).fillna(0).astype(int)

    df.drop_duplicates(inplace=True)
    df.to_sql("accidents_clean", engine, if_exists="replace", index=False)
    print(f"Shape after:  {df.shape}")
    print(" accidents_clean saved.")
    df.head(3)


  Dropped 1 records outside Bangalore bbox.
Shape after:  (9, 15)
 accidents_clean saved.


,date,time,location,latitude,longitude,severity,vehicles,casualties,violation_type,fatalities,injuries,year,month,hour,severity_score
0,2024-01-15,08:30:00,Silk Board Junction,12.9176,77.6227,High,2,1,Speeding,0,1,2024,1,NaN,0
1,2024-02-10,18:45:00,Marathahalli Bridge,12.9571,77.7006,Medium,3,0,Tailgating,0,0,2024,2,NaN,0
2,2024-03-05,09:15:00,Hebbal Flyover,13.0450,77.5943,Critical,2,2,Drunk Driving,1,1,2024,3,NaN,0


###  4. BMTC Stops — Clean


In [8]:
df = load("bmtc_stops")
print(f"Shape before: {df.shape}")


Shape before: (10, 4)


In [9]:
# Only write to bmtc_stops_clean if enrich_data.py hasn't already populated it
with engine.connect() as _c:
    _n = _c.execute(text("SELECT COUNT(*) FROM bmtc_stops_clean")).scalar()

if _n >= 100:
    print(f"bmtc_stops_clean already has {_n} enriched rows — skipping overwrite.")
    print("To reset, run: enrich_data.py")
else:
    bbox = BANGALORE_BBOX
    # Keep only stops inside Bangalore
    df = df[
        df["latitude"].between(bbox["lat_min"], bbox["lat_max"]) &
        df["longitude"].between(bbox["lon_min"], bbox["lon_max"])
    ]
    df.drop_duplicates(subset="stop_id", inplace=True)
    df["stop_name"] = df["stop_name"].str.strip().str.title()

    df.to_sql("bmtc_stops_clean", engine, if_exists="replace", index=False)
    print(f"Shape after:  {df.shape}")
    print(" bmtc_stops_clean saved.")
    df.head(3)


Shape after:  (9, 4)
 bmtc_stops_clean saved.


,stop_id,stop_name,latitude,longitude
0,1001,Silk Board Junction,12.9176,77.6227
1,1002,Marathahalli Bridge,12.9571,77.7006
2,1003,Hebbal Flyover,13.0450,77.5943


###  5. Road Network — Clean


In [10]:
df = load("road_network")
print(f"Shape before: {df.shape}")
print(df.isnull().sum())

Shape before: (93161, 7)
osmid               0
name            73192
highway             0
length_m            0
lanes           88445
maxspeed        91117
geometry_wkt        0
dtype: int64


In [11]:
# Fill missing road names and clean highway type
df["name"] = df["name"].replace("nan", None).fillna("Unnamed Road")
df["highway"] = df["highway"].replace("nan", None).fillna("unclassified")

# Fix speed limits: fill missing based on road type
default_speeds = {
    "motorway": 100, "trunk": 80, "primary": 60,
    "secondary": 50, "tertiary": 40, "residential": 30,
    "unclassified": 30,
}

# Convert maxspeed to numeric if possible
df["maxspeed"] = pd.to_numeric(df["maxspeed"], errors="coerce").fillna(0)

for road_type, speed in default_speeds.items():
    mask = (df["highway"].astype(str).str.contains(road_type, na=False)) & (df["maxspeed"] == 0)
    df.loc[mask, "maxspeed"] = speed

df.to_sql("road_network_clean", engine, if_exists="replace", index=False)
print(f"Shape after:  {df.shape}")
print(" road_network_clean saved.")
df.head(3)


Shape after:  (93161, 7)
 road_network_clean saved.


,osmid,name,highway,length_m,lanes,maxspeed,geometry_wkt
0,32261256,2nd Main Road,residential,244.091315,2,30.0,"LINESTRING (77.598721 12.910542, 77.598478 12...."
1,1367650597,9th Cross Road,secondary,29.723619,2,50.0,"LINESTRING (77.598721 12.910542, 77.598994 12...."
2,1367650758,2nd Main Road,residential,5.961520,None,30.0,"LINESTRING (77.598721 12.910542, 77.598726 12...."


###  Cleaning Summary


In [12]:
print("=== Cleaned Table Counts ===")
tables = ["traffic_speeds_clean", "weather_clean", "accidents_clean", "bmtc_stops_clean", "road_network_clean"]
with engine.connect() as conn:
    for t in tables:
        try:
            count = conn.execute(text(f"SELECT COUNT(*) FROM {t}")).scalar()
            print(f"  {t:<30} {count:>8,} rows")
        except Exception as e:
            print(f"  {t:<30} ERROR: {e}")
print("\n Data cleaning complete. Proceed to Notebook 3 — Analysis.")


=== Cleaned Table Counts ===
  traffic_speeds_clean              7,250 rows
  weather_clean                    20,640 rows
  accidents_clean                       9 rows
  bmtc_stops_clean                      9 rows
  road_network_clean               93,161 rows

 Data cleaning complete. Proceed to Notebook 3 — Analysis.
